# 第13章: プロパティとメソッドの詳細 - 解答集

この解答集では、課題の解答と詳細な解説を提供します。

**重要**: 自分で解いてから解答を見るようにしてください！

---
## 課題1: バリューオブジェクト（基礎レベル）- 解答

### 解答コード

In [ ]:
<?php

/**
 * Emailクラス - メールアドレスを表すバリューオブジェクト
 */
class Email {
    private string $value;
    
    public function __construct(string $email) {
        // バリデーション
        if (!filter_var($email, FILTER_VALIDATE_EMAIL)) {
            throw new InvalidArgumentException("無効なメールアドレス形式: {$email}");
        }
        $this->value = $email;
    }
    
    public function getValue(): string {
        return $this->value;
    }
    
    public function getDomain(): string {
        return substr(strstr($this->value, '@'), 1);
    }
    
    public function getLocalPart(): string {
        return strstr($this->value, '@', true);
    }
    
    public function equals(Email $other): bool {
        return strtolower($this->value) === strtolower($other->value);
    }
    
    public function __toString(): string {
        return $this->value;
    }
}

/**
 * PhoneNumberクラス - 電話番号を表すバリューオブジェクト
 */
class PhoneNumber {
    private string $number;
    private string $countryCode;
    
    public function __construct(string $number, string $countryCode = '+81') {
        // 数字とハイフンのみ抽出
        $cleanNumber = preg_replace('/[^0-9]/', '', $number);
        
        // バリデーション（10桁または11桁）
        if (strlen($cleanNumber) < 10 || strlen($cleanNumber) > 11) {
            throw new InvalidArgumentException("無効な電話番号: {$number}");
        }
        
        $this->number = $cleanNumber;
        $this->countryCode = $countryCode;
    }
    
    public function getFormatted(): string {
        if (strlen($this->number) === 11) {
            // 携帯電話: 090-1234-5678
            return preg_replace('/(\d{3})(\d{4})(\d{4})/', '$1-$2-$3', $this->number);
        } else {
            // 固定電話: 03-1234-5678
            return preg_replace('/(\d{2})(\d{4})(\d{4})/', '$1-$2-$3', $this->number);
        }
    }
    
    public function getInternationalFormat(): string {
        $formatted = $this->getFormatted();
        // 先頭の0を削除
        $withoutLeadingZero = preg_replace('/^0/', '', $formatted);
        return $this->countryCode . '-' . $withoutLeadingZero;
    }
    
    public function getNumber(): string { return $this->number; }
    public function getCountryCode(): string { return $this->countryCode; }
}

/**
 * Moneyクラス - 金額を表す不変オブジェクト
 */
class Money {
    public function __construct(
        private readonly int $amount,
        private readonly string $currency = 'JPY'
    ) {
        if ($amount < 0) {
            throw new InvalidArgumentException("金額は0以上である必要があります");
        }
    }
    
    public function add(Money $other): Money {
        if ($this->currency !== $other->currency) {
            throw new InvalidArgumentException("通貨が異なります");
        }
        return new Money($this->amount + $other->amount, $this->currency);
    }
    
    public function subtract(Money $other): Money {
        if ($this->currency !== $other->currency) {
            throw new InvalidArgumentException("通貨が異なります");
        }
        $newAmount = $this->amount - $other->amount;
        if ($newAmount < 0) {
            throw new InvalidArgumentException("金額が負になります");
        }
        return new Money($newAmount, $this->currency);
    }
    
    public function multiply(float $factor): Money {
        return new Money((int)round($this->amount * $factor), $this->currency);
    }
    
    public function format(): string {
        return match($this->currency) {
            'JPY' => '¥' . number_format($this->amount),
            'USD' => '$' . number_format($this->amount / 100, 2),
            'EUR' => '€' . number_format($this->amount / 100, 2, ',', ' '),
            default => $this->amount . ' ' . $this->currency
        };
    }
    
    public function getAmount(): int { return $this->amount; }
    public function getCurrency(): string { return $this->currency; }
}

// 動作確認
echo "=== Emailクラス ===\n";
$email1 = new Email("user@example.com");
echo "メールアドレス: {$email1->getValue()}\n";
echo "ドメイン: {$email1->getDomain()}\n";
echo "ローカルパート: {$email1->getLocalPart()}\n";

$email2 = new Email("USER@EXAMPLE.COM");
echo "\n同じアドレスか: " . ($email1->equals($email2) ? "はい" : "いいえ") . "\n";

echo "\n=== PhoneNumberクラス ===\n";
$phone = new PhoneNumber("090-1234-5678", "+81");
echo "国内形式: " . $phone->getFormatted() . "\n";
echo "国際形式: " . $phone->getInternationalFormat() . "\n";

echo "\n=== Moneyクラス ===\n";
$money1 = new Money(1000);
$money2 = new Money(500);

$sum = $money1->add($money2);
echo "合計: " . $sum->format() . "\n";

$doubled = $money1->multiply(2);
echo "2倍: " . $doubled->format() . "\n";

?>

### 解説

#### 1. バリューオブジェクトパターン

**バリューオブジェクトとは**:
- 値そのものを表現するオブジェクト
- コンストラクタでバリデーション
- 不変性（変更不可）を持つことが多い
- equals()で値の比較

**利点**:
- 無効な値が存在しないことを保証
- コードの意図が明確
- テストが容易

#### 2. readonlyプロパティ（PHP 8.2+）

```php
public function __construct(
    private readonly int $amount,
    private readonly string $currency = 'JPY'
) {}
```

**readonlyの特徴**:
- 一度値を設定したら変更不可
- 不変クラスを簡単に実装
- コンパイル時にチェック

#### 3. マジックメソッド __toString()

```php
public function __toString(): string {
    return $this->value;
}

// 使い方
echo $email;  // 自動的に__toString()が呼ばれる
```

**ポイント**:
- 文字列を期待する場所で自動的に呼ばれる
- 必ずstringを返す必要がある
- 例外を投げてはいけない

#### 4. 実務での応用

**ドメイン駆動設計（DDD）**:
```php
// 悪い例：プリミティブな値
function sendEmail(string $to) {
    // バリデーションが必要
}

// 良い例：バリューオブジェクト
function sendEmail(Email $to) {
    // 既にバリデーション済み
}
```

---
## 課題2: ユーザークラス（中級レベル）- 解答

### 解答コード

In [ ]:
<?php

/**
 * Userクラス - ユーザーを表す
 */
class User {
    private ?int $id;
    private string $name;
    private string $email;
    private int $age;
    private string $password;
    private DateTime $createdAt;
    private DateTime $updatedAt;
    
    public function __construct(
        ?int $id,
        string $name,
        string $email,
        int $age,
        string $password
    ) {
        $this->id = $id;
        $this->createdAt = new DateTime();
        $this->updatedAt = new DateTime();
        
        // セッター経由でバリデーション
        $this->setName($name);
        $this->setEmail($email);
        $this->setAge($age);
        $this->setPassword($password);
    }
    
    // ゲッター
    public function getId(): ?int {
        return $this->id;
    }
    
    public function getName(): string {
        return $this->name;
    }
    
    public function getEmail(): string {
        return $this->email;
    }
    
    public function getAge(): int {
        return $this->age;
    }
    
    // セッター（バリデーション付き）
    public function setName(string $name): void {
        $trimmed = trim($name);
        if (empty($trimmed)) {
            throw new InvalidArgumentException("名前は空にできません");
        }
        if (mb_strlen($trimmed) > 50) {
            throw new InvalidArgumentException("名前は50文字以下である必要があります");
        }
        $this->name = $trimmed;
        $this->markAsUpdated();
    }
    
    public function setEmail(string $email): void {
        if (!filter_var($email, FILTER_VALIDATE_EMAIL)) {
            throw new InvalidArgumentException("無効なメールアドレス形式");
        }
        $this->email = $email;
        $this->markAsUpdated();
    }
    
    public function setAge(int $age): void {
        if ($age < 18 || $age > 120) {
            throw new InvalidArgumentException("年齢は18歳から120歳の範囲である必要があります");
        }
        $this->age = $age;
        $this->markAsUpdated();
    }
    
    public function setPassword(string $password): void {
        if (strlen($password) < 8) {
            throw new InvalidArgumentException("パスワードは8文字以上である必要があります");
        }
        $this->password = password_hash($password, PASSWORD_DEFAULT);
        $this->markAsUpdated();
    }
    
    public function verifyPassword(string $password): bool {
        return password_verify($password, $this->password);
    }
    
    public function isAdult(): bool {
        return $this->age >= 18;
    }
    
    public function getCreatedAt(): DateTime {
        return $this->createdAt;
    }
    
    public function getUpdatedAt(): DateTime {
        return $this->updatedAt;
    }
    
    public function toArray(): array {
        return [
            'id' => $this->id,
            'name' => $this->name,
            'email' => $this->email,
            'age' => $this->age,
            'created_at' => $this->createdAt->format('Y-m-d H:i:s'),
            'updated_at' => $this->updatedAt->format('Y-m-d H:i:s')
        ];
    }
    
    private function markAsUpdated(): void {
        $this->updatedAt = new DateTime();
    }
}

// 動作確認
echo "=== ユーザー作成 ===\n";
$user = new User(null, "田中太郎", "tanaka@example.com", 25, "password123");

echo "名前: " . $user->getName() . "\n";
echo "メール: " . $user->getEmail() . "\n";
echo "年齢: " . $user->getAge() . "歳\n";
echo "成人: " . ($user->isAdult() ? "はい" : "いいえ") . "\n";

echo "\n=== パスワード検証 ===\n";
echo "正しいパスワード: " . ($user->verifyPassword("password123") ? "OK" : "NG") . "\n";
echo "間違ったパスワード: " . ($user->verifyPassword("wrong") ? "OK" : "NG") . "\n";

echo "\n=== バリデーション ===\n";
try {
    $user->setAge(15);
} catch (InvalidArgumentException $e) {
    echo "年齢エラー: {$e->getMessage()}\n";
}

try {
    $user->setEmail("invalid-email");
} catch (InvalidArgumentException $e) {
    echo "メールエラー: {$e->getMessage()}\n";
}

echo "\n=== toArray() ===\n";
print_r($user->toArray());

?>

### 解説

#### 1. ゲッターとセッター

**なぜ使うのか**:
```php
// 悪い例：publicプロパティ
public $age;
$user->age = -5;  // 不正な値を代入できてしまう

// 良い例：private + セッター
private int $age;
$user->setAge(-5);  // 例外が発生
```

#### 2. バリデーション

**セッターでのチェック**:
```php
public function setAge(int $age): void {
    if ($age < 18 || $age > 120) {
        throw new InvalidArgumentException(...);
    }
    $this->age = $age;
}
```

**バリデーションのタイミング**:
- コンストラクタ（初期化時）
- セッター（変更時）
- 常に整合性を保つ

#### 3. パスワードハッシュ

**PHPのパスワード関数**:
```php
// ハッシュ化
$hash = password_hash($password, PASSWORD_DEFAULT);

// 検証
if (password_verify($input, $hash)) {
    // 一致
}
```

**ポイント**:
- 平文で保存してはいけない
- `password_hash()` が自動的にソルトを生成
- `password_verify()` で安全に比較

#### 4. 更新日時の自動追跡

```php
private function markAsUpdated(): void {
    $this->updatedAt = new DateTime();
}

// セッターで呼び出す
public function setName(string $name): void {
    $this->name = $name;
    $this->markAsUpdated();  // 最終更新時刻を記録
}
```

---
## 課題3: QueryBuilder（上級レベル）- 解答

### 解答コード

In [ ]:
<?php

/**
 * QueryBuilderクラス - SQLクエリをビルドする
 */
class QueryBuilder {
    private array $select;
    private ?string $from;
    private array $where;
    private ?string $orderBy;
    private ?string $orderDirection;
    private ?int $limit;
    private ?int $offset;
    
    public function __construct() {
        $this->reset();
    }
    
    public function select(array $columns = ['*']): self {
        $this->select = $columns;
        return $this;
    }
    
    public function from(string $table): self {
        $this->from = $table;
        return $this;
    }
    
    public function where(string $condition): self {
        $this->where[] = $condition;
        return $this;
    }
    
    public function orderBy(string $column, string $direction = 'ASC'): self {
        $direction = strtoupper($direction);
        if (!in_array($direction, ['ASC', 'DESC'])) {
            throw new InvalidArgumentException("方向はASCまたはDESCである必要があります");
        }
        $this->orderBy = $column;
        $this->orderDirection = $direction;
        return $this;
    }
    
    public function limit(int $count): self {
        if ($count < 0) {
            throw new InvalidArgumentException("LIMITは0以上である必要があります");
        }
        $this->limit = $count;
        return $this;
    }
    
    public function offset(int $count): self {
        if ($count < 0) {
            throw new InvalidArgumentException("OFFSETは0以上である必要があります");
        }
        $this->offset = $count;
        return $this;
    }
    
    public function build(): string {
        $query = [];
        
        // SELECT句
        if (empty($this->select)) {
            throw new RuntimeException("SELECT句が指定されていません");
        }
        $query[] = 'SELECT ' . implode(', ', $this->select);
        
        // FROM句
        if ($this->from === null) {
            throw new RuntimeException("FROM句が指定されていません");
        }
        $query[] = 'FROM ' . $this->from;
        
        // WHERE句
        if (!empty($this->where)) {
            $query[] = 'WHERE ' . implode(' AND ', $this->where);
        }
        
        // ORDER BY句
        if ($this->orderBy !== null) {
            $query[] = "ORDER BY {$this->orderBy} {$this->orderDirection}";
        }
        
        // LIMIT句
        if ($this->limit !== null) {
            $query[] = "LIMIT {$this->limit}";
        }
        
        // OFFSET句
        if ($this->offset !== null) {
            $query[] = "OFFSET {$this->offset}";
        }
        
        return implode(' ', $query);
    }
    
    public function reset(): void {
        $this->select = [];
        $this->from = null;
        $this->where = [];
        $this->orderBy = null;
        $this->orderDirection = 'ASC';
        $this->limit = null;
        $this->offset = null;
    }
    
    /**
     * マジックメソッド - 動的なwhereメソッド
     * 例: whereAge(18), whereName('John')
     */
    public function __call(string $name, array $arguments) {
        // 'where'で始まるかチェック
        if (str_starts_with($name, 'where')) {
            // 'where'以降を取得（例: 'whereAge' -> 'Age'）
            $column = substr($name, 5);
            // キャメルケースをスネークケースに変換

            $column = strtolower(preg_replace('/([A-Z])/', '_$1', lcfirst($column)));
            
            if (empty($arguments)) {
                throw new InvalidArgumentException("whereメソッドには引数が必要です");
            }
            
            $value = $arguments[0];
            
            // 値の型に応じて条件を生成
            if (is_string($value)) {
                $condition = "{$column} = '{$value}'";
            } elseif (is_null($value)) {
                $condition = "{$column} IS NULL";
            } else {
                $condition = "{$column} = {$value}";
            }
            
            $this->where[] = $condition;
            return $this;
        }
        
        throw new BadMethodCallException("メソッド {$name} は存在しません");
    }
    
    // ゲッター
    public function getSelect(): array { return $this->select; }
    public function getFrom(): ?string { return $this->from; }
    public function getWhere(): array { return $this->where; }
}

// 動作確認
echo "=== 基本的なクエリ ===\n";
$query = (new QueryBuilder())
    ->select(['id', 'name', 'email'])
    ->from('users')
    ->where('age >= 18')
    ->orderBy('created_at', 'DESC')
    ->limit(10);

echo $query->build() . "\n";

echo "\n=== 複数のWHERE句 ===\n";
$query2 = (new QueryBuilder())
    ->select()
    ->from('products')
    ->where('price > 1000')
    ->where('stock > 0')
    ->where('category = "electronics"')
    ->orderBy('price');

echo $query2->build() . "\n";

echo "\n=== マジックメソッドの使用 ===\n";
$query3 = (new QueryBuilder())
    ->select()
    ->from('users')
    ->whereAge(25)
    ->whereStatus('active');

echo $query3->build() . "\n";

echo "\n=== ページネーション ===\n";
$page = 2;
$perPage = 20;
$query4 = (new QueryBuilder())
    ->select()
    ->from('articles')
    ->orderBy('id')
    ->limit($perPage)
    ->offset(($page - 1) * $perPage);

echo $query4->build() . "\n";

?>

### 解説

#### 1. メソッドチェーン

**`$this` を返す**:
```php
public function select(array $columns): self {
    $this->select = $columns;
    return $this;  // 自分自身を返す
}

// 使い方
$query->select()->from()->where();
```

**利点**:
- コードが読みやすい
- 流れるようなインターフェース
- Laravel、Doctrineなどで使用

#### 2. マジックメソッド __call()

**動的メソッドの呼び出し**:
```php
public function __call(string $name, array $arguments) {
    if (str_starts_with($name, 'where')) {
        $column = substr($name, 5);  // 'whereAge' -> 'Age'
        // キャメルケースをスネークケースに
        $column = strtolower(preg_replace('/([A-Z])/', '_$1', lcfirst($column)));
        // 条件を生成
        $this->where[] = "{$column} = {$arguments[0]}";
        return $this;
    }
}

// 使い方
$query->whereAge(25);  // 動的に生成
```

**パターン**:
- プレフィックスで判断（where, get, set）
- 文字列操作で処理
- 柔軟なAPI設計

#### 3. LaravelのQueryBuilderとの比較

**Laravel風の実装**:
```php
// Laravel
DB::table('users')
    ->select(['id', 'name'])
    ->where('age', '>=', 18)
    ->orderBy('created_at')
    ->limit(10)
    ->get();

// 私たちの実装
(new QueryBuilder())
    ->select(['id', 'name'])
    ->from('users')
    ->where('age >= 18')
    ->orderBy('created_at')
    ->limit(10)
    ->build();
```

---
## ベストプラクティスまとめ

### 1. アクセス修飾子の使い分け

| 修飾子 | 使い道 | 例 |
|--------|--------|-----|
| public | 外部に公開するAPI | ゲッター、ビジネスロジック |
| protected | サブクラスから利用 | 共通メソッド |
| private | クラス内のみ | ヘルパーメソッド、プロパティ |

### 2. 不変クラスの設計

```php
class Money {
    public function __construct(
        private readonly int $amount  // 変更不可
    ) {}
    
    public function add(Money $other): Money {
        return new Money($this->amount + $other->amount);  // 新しいインスタンス
    }
}
```

### 3. バリデーションの原則

**早期失敗（Fail Fast）**:
```php
public function setEmail(string $email): void {
    if (!filter_var($email, FILTER_VALIDATE_EMAIL)) {
        throw new InvalidArgumentException(...);  // すぐに例外
    }
    $this->email = $email;
}
```

### 4. メソッドチェーンの実装

**ルール**:
1. `self` または `this` を返す
2. メソッド名は動詞で始める
3. 設定系メソッドに適している

### 5. Web開発での応用

**Laravel Eloquentモデル**:
```php
class User extends Model {
    protected $fillable = ['name', 'email'];  // 対象プロパティ
    
    // アクセサ
    public function getFullNameAttribute(): string {
        return "{$this->first_name} {$this->last_name}";
    }
    
    // ミューテータ
    public function setPasswordAttribute(string $password): void {
        $this->attributes['password'] = Hash::make($password);
    }
}
```

### 🎯 次のステップ

第13章を完了しました！次は第14章で、継承とポリモーフィズムについて学びます。

**次章の予習**:
- extendsキーワード
- オーバーライド
- parent::の使い方
- 抽象クラスとインターフェース
- ポリモーフィズム